# Notebook 5 — MLlib training & evaluation

Time-aware split: train `week_year < 2021`, test `week_year >= 2021`. Pipeline: VectorAssembler → StandardScaler → RandomForestClassifier.


In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("."))
import hdfs_paths as hp

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession.builder.appName("MusicTrend_05_Model")
    .config("spark.sql.shuffle.partitions", hp.SHUFFLE_PARTITIONS)
    .getOrCreate()
)

FEATURE_COLS = [
    "plays_1d",
    "plays_7d",
    "plays_28d",
    "growth_rate_7d",
    "stream_velocity",
    "region_spread",
    "total_streams",
    "tempo",
    "energy",
    "loudness",
    "danceability",
]
LABEL_COL = "charted"

features = spark.read.parquet(hp.PROCESSED_FEATURES).fillna(0, subset=FEATURE_COLS)

train = features.filter(col("week_year") < 2021)
test = features.filter(col("week_year") >= 2021)

print(train.count(), test.count())
train.groupBy(LABEL_COL).count().show()
test.groupBy(LABEL_COL).count().show()


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

assembler = VectorAssembler(
    inputCols=FEATURE_COLS,
    outputCol="raw_features",
    handleInvalid="keep",
)
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withStd=True,
    withMean=True,
)
rf = RandomForestClassifier(
    labelCol=LABEL_COL,
    featuresCol="features",
    numTrees=100,
    maxDepth=8,
    seed=42,
)

pipeline = Pipeline(stages=[assembler, scaler, rf])
model = pipeline.fit(train)


In [ ]:
predictions = model.transform(test)

auc_eval = BinaryClassificationEvaluator(labelCol=LABEL_COL, metricName="areaUnderROC")
auc = auc_eval.evaluate(predictions)
print(f"AUC-ROC: {auc:.4f}")

f1_eval = MulticlassClassificationEvaluator(labelCol=LABEL_COL, metricName="f1")
f1 = f1_eval.evaluate(predictions)
print(f"F1 Score: {f1:.4f}")

predictions.groupBy(LABEL_COL, "prediction").count().show()

rf_model = model.stages[-1]
importances = list(zip(FEATURE_COLS, rf_model.featureImportances.toArray()))
importances.sort(key=lambda x: x[1], reverse=True)
for feat, imp in importances:
    print(f"  {feat}: {imp:.4f}")

model.write().overwrite().save(hp.MODEL_RF)
print("Saved model to", hp.MODEL_RF)


## Outputs confirmed

- Metrics printed; confusion table shown.
- Model saved to HDFS `models/rf_model`.
